# IPL Data Analysis Pipeline
### End-to-End: Ingest → Clean → Transform → Analyze → Report → Export

## Stage 1: Data Ingestion

In [78]:
import pandas as pd
import numpy as np


In [ ]:

deliveries_df = pd.read_csv('deliveries.csv')
matches_df    = pd.read_csv('matches.csv')

print(f'deliveries_df shape : {deliveries_df.shape}')
print(f'matches_df shape    : {matches_df.shape}')

deliveries_df shape : (260920, 17)
matches_df shape    : (1095, 20)


In [80]:
print('=== deliveries_df columns ===')
print(deliveries_df.columns.tolist())
print('\n=== matches_df columns ===')
print(matches_df.columns.tolist())

=== deliveries_df columns ===
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']

=== matches_df columns ===
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']


In [81]:
print('=== deliveries_df dtypes ===')
print(deliveries_df.dtypes)
print('\n=== matches_df dtypes ===')
print(matches_df.dtypes)

=== deliveries_df dtypes ===
match_id            int64
inning              int64
batting_team          str
bowling_team          str
over                int64
ball                int64
batter                str
bowler                str
non_striker           str
batsman_runs        int64
extra_runs          int64
total_runs          int64
extras_type           str
is_wicket           int64
player_dismissed      str
dismissal_kind        str
fielder               str
dtype: object

=== matches_df dtypes ===
id                   int64
season                 str
city                   str
date                   str
match_type             str
player_of_match        str
venue                  str
team1                  str
team2                  str
toss_winner            str
toss_decision          str
winner                 str
result                 str
result_margin      float64
target_runs        float64
target_overs       float64
super_over             str
method                 str
um

In [82]:
deliveries_df.head(3)

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN


In [83]:
matches_df.head(3)

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar


## Stage 2: Data Cleaning & Validation

In [84]:
print('=== Missing values in deliveries_df ===')
print(deliveries_df.isnull().sum()[deliveries_df.isnull().sum() > 0])
print('\n=== Missing values in matches_df ===')
print(matches_df.isnull().sum()[matches_df.isnull().sum() > 0])

=== Missing values in deliveries_df ===
extras_type         246795
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

=== Missing values in matches_df ===
city                 51
player_of_match       5
winner                5
result_margin        19
target_runs           3
target_overs          3
method             1074
dtype: int64


In [ ]:
run_cols = ['batsman_runs', 'extra_runs', 'total_runs', 'wide_runs',
            'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs']
for col in run_cols:
    if col in deliveries_df.columns:
        deliveries_df[col] = deliveries_df[col].fillna(0).astype(int)

if 'player_dismissed' in deliveries_df.columns:
    deliveries_df['player_dismissed'] = deliveries_df['player_dismissed'].fillna('not_out')
if 'dismissal_kind' in deliveries_df.columns:
    deliveries_df['dismissal_kind'] = deliveries_df['dismissal_kind'].fillna('none')
if 'fielder' in deliveries_df.columns:
    deliveries_df['fielder'] = deliveries_df['fielder'].fillna('none')

for col in ['player_of_match', 'umpire1', 'umpire2', 'umpire3', 'city', 'winner', 'result']:
    if col in matches_df.columns:
        matches_df[col] = matches_df[col].fillna('Unknown')

print('Cleaning done ')

Cleaning done 


In [ ]:
del_ids  = set(deliveries_df['match_id'].unique())
mat_ids  = set(matches_df['id'].unique())
missing  = del_ids - mat_ids
print(f'Match IDs in deliveries but NOT in matches: {len(missing)}')
if missing:
    print('Sample missing IDs:', list(missing)[:5])

Match IDs in deliveries but NOT in matches: 0


In [ ]:
deliveries_df['match_id'] = deliveries_df['match_id'].astype(int)
matches_df['id']          = matches_df['id'].astype(int)

if 'season' in matches_df.columns:
    matches_df['season'] = matches_df['season'].astype(str)

print('Validation complete ')
print(f'deliveries_df shape after cleaning: {deliveries_df.shape}')
print(f'matches_df shape after cleaning   : {matches_df.shape}')

Validation complete 
deliveries_df shape after cleaning: (260920, 17)
matches_df shape after cleaning   : (1095, 20)


## Stage 3: Data Transformation

In [ ]:
merged_df = deliveries_df.merge(matches_df, left_on='match_id', right_on='id', how='left')
print(f'Merged DataFrame shape: {merged_df.shape}')
merged_df.head(2)

Merged DataFrame shape: (260920, 37)


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen


In [ ]:
merged_df.columns = [c.strip().lower().replace(' ', '_') for c in merged_df.columns]

if 'total_runs' not in merged_df.columns:
    merged_df['total_runs'] = merged_df['batsman_runs'] + merged_df.get('extra_runs', 0)

merged_df['is_four'] = (merged_df['batsman_runs'] == 4).astype(int)
merged_df['is_six']  = (merged_df['batsman_runs'] == 6).astype(int)

merged_df['is_dot'] = ((merged_df['batsman_runs'] == 0) &
                        (merged_df.get('extra_runs', pd.Series(0, index=merged_df.index)) == 0)).astype(int)

print('Transformation complete')

Transformation complete


## Stage 4: Core Analysis

### 4.1 Total Runs per Match

In [90]:
runs_per_match = (
    merged_df.groupby('match_id')['total_runs']
    .sum()
    .reset_index()
    .rename(columns={'total_runs': 'total_runs_in_match'})
    .sort_values('total_runs_in_match', ascending=False)
)
print(runs_per_match.head(10).to_string(index=False))

 match_id  total_runs_in_match
  1426268                  549
  1422126                  523
  1426280                  523
  1426281                  504
   419137                  469
  1426273                  465
  1136604                  459
  1359512                  458
  1082641                  453
  1216527                  449


### 4.2 Runs per Team per Match

In [ ]:
bat_team_col = 'batting_team' if 'batting_team' in merged_df.columns else 'team1'

team_scores = (
    merged_df.groupby(['match_id', bat_team_col])['total_runs']
    .sum()
    .reset_index()
    .rename(columns={bat_team_col: 'team', 'total_runs': 'team_runs'})
    .sort_values(['match_id', 'team_runs'], ascending=[True, False])
)
print(team_scores.head(10).to_string(index=False))

 match_id                        team  team_runs
   335982       Kolkata Knight Riders        222
   335982 Royal Challengers Bangalore         82
   335983         Chennai Super Kings        240
   335983             Kings XI Punjab        207
   335984            Delhi Daredevils        132
   335984            Rajasthan Royals        129
   335985 Royal Challengers Bangalore        166
   335985              Mumbai Indians        165
   335986       Kolkata Knight Riders        112
   335986             Deccan Chargers        110


### 4.3 Top 10 Batters by Total Runs

In [ ]:
bat_col = 'batter' if 'batter' in merged_df.columns else 'batsman'

top_batters = (
    merged_df.groupby(bat_col)['batsman_runs']
    .sum()
    .reset_index()
    .rename(columns={bat_col: 'batter', 'batsman_runs': 'total_runs'})
    .sort_values('total_runs', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_batters.index += 1
print(top_batters.to_string())

            batter  total_runs
1          V Kohli        8014
2         S Dhawan        6769
3        RG Sharma        6630
4        DA Warner        6567
5         SK Raina        5536
6         MS Dhoni        5243
7   AB de Villiers        5181
8         CH Gayle        4997
9       RV Uthappa        4954
10      KD Karthik        4843


### 4.4 Strike Rate of Batters

In [93]:
strike_rate = (
    merged_df.groupby(bat_col)
    .agg(total_runs=('batsman_runs', 'sum'),
         balls_faced=('batsman_runs', 'count'))
    .reset_index()
    .rename(columns={bat_col: 'batter'})
)
strike_rate['strike_rate'] = (strike_rate['total_runs'] / strike_rate['balls_faced'] * 100).round(2)
strike_rate = strike_rate[strike_rate['balls_faced'] >= 200].sort_values('strike_rate', ascending=False)
print(strike_rate.head(10).to_string(index=False))

        batter  total_runs  balls_faced  strike_rate
       PD Salt         653          385       169.61
      T Stubbs         405          239       169.46
       TM Head         772          458       168.56
    AD Russell        2488         1515       164.22
     H Klaasen         993          613       161.99
      TH David         659          417       158.03
     SP Narine        1534          984       155.89
Shashank Singh         423          272       155.51
      N Pooran        1769         1143       154.77
LS Livingstone         939          609       154.19


### 4.5 Top 10 Bowlers by Economy Rate

In [94]:
bowl_col = 'bowler'

economy = (
    merged_df.groupby(bowl_col)
    .agg(runs_conceded=('total_runs', 'sum'),
         balls_bowled=(bowl_col, 'count'))
    .reset_index()
)
economy['overs_bowled'] = economy['balls_bowled'] / 6
economy['economy_rate'] = (economy['runs_conceded'] / economy['overs_bowled']).round(2)
economy = economy[economy['balls_bowled'] >= 120].sort_values('economy_rate')
economy = economy.rename(columns={bowl_col: 'bowler'})
print(economy.head(10).to_string(index=False))

         bowler  runs_conceded  balls_bowled  overs_bowled  economy_rate
  Sohail Tanvir            275           265     44.166667          6.23
     A Chandila            245           234     39.000000          6.28
     FH Edwards            160           150     25.000000          6.40
SMSM Senanayake            211           195     32.500000          6.49
     SM Pollock            307           280     46.666667          6.58
       A Kumble           1089           983    163.833333          6.65
     GD McGrath            366           329     54.833333          6.67
 M Muralitharan           1765          1581    263.500000          6.70
       IS Sodhi            204           182     30.333333          6.73
        J Yadav            447           398     66.333333          6.74


### 4.6 Most Consistent Batters (High Average, Min 10 Matches)

In [95]:
consistent_batters = (
    merged_df.groupby([bat_col, 'match_id'])['batsman_runs']
    .sum()
    .reset_index()
    .groupby(bat_col)
    .agg(matches_played=('match_id', 'nunique'),
         total_runs=('batsman_runs', 'sum'))
    .reset_index()
)
consistent_batters['avg_runs_per_match'] = (
    consistent_batters['total_runs'] / consistent_batters['matches_played']
).round(2)
consistent_batters = (
    consistent_batters[consistent_batters['matches_played'] >= 10]
    .sort_values('avg_runs_per_match', ascending=False)
    .rename(columns={bat_col: 'batter'})
    .head(10)
    .reset_index(drop=True)
)
consistent_batters.index += 1
print(consistent_batters.to_string())

             batter  matches_played  total_runs  avg_runs_per_match
1         DP Conway              22         924               42.00
2   B Sai Sudharsan              25        1034               41.36
3          KL Rahul             122        4689               38.43
4       LMP Simmons              29        1079               37.21
5        RD Gaikwad              65        2380               36.62
6          SE Marsh              69        2489               36.07
7           HM Amla              16         577               36.06
8         DA Warner             184        6567               35.69
9          CH Gayle             141        4997               35.44
10        ML Hayden              32        1107               34.59


### 4.7 Highest Individual Score in a Match

In [96]:
highest_individual = (
    merged_df.groupby([bat_col, 'match_id'])['batsman_runs']
    .sum()
    .reset_index()
    .rename(columns={bat_col: 'batter', 'batsman_runs': 'runs_in_match'})
    .sort_values('runs_in_match', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
highest_individual.index += 1
print(highest_individual.to_string())

            batter  match_id  runs_in_match
1         CH Gayle    598027            175
2      BB McCullum    335982            158
3        Q de Kock   1304112            140
4   AB de Villiers    829795            133
5         KL Rahul   1216510            132
6     Shubman Gill   1370352            129
7   AB de Villiers    980987            129
8         CH Gayle    548372            128
9          RR Pant   1136602            128
10         M Vijay    419137            127


### 4.8 Boundary Analysis

In [97]:
total_fours = merged_df['is_four'].sum()
total_sixes = merged_df['is_six'].sum()
print(f'Total 4s: {total_fours}')
print(f'Total 6s: {total_sixes}')

boundaries_by_player = (
    merged_df.groupby(bat_col)
    .agg(fours=('is_four', 'sum'), sixes=('is_six', 'sum'))
    .reset_index()
    .rename(columns={bat_col: 'batter'})
)
boundaries_by_player['total_boundaries'] = boundaries_by_player['fours'] + boundaries_by_player['sixes']
boundaries_by_player = boundaries_by_player.sort_values('total_boundaries', ascending=False).head(10).reset_index(drop=True)
boundaries_by_player.index += 1
print('\nTop 10 players by boundaries:')
print(boundaries_by_player.to_string())

Total 4s: 29850
Total 6s: 13051

Top 10 players by boundaries:
            batter  fours  sixes  total_boundaries
1          V Kohli    708    273               981
2         S Dhawan    768    153               921
3        DA Warner    663    236               899
4        RG Sharma    599    281               880
5         CH Gayle    408    359               767
6         SK Raina    506    204               710
7   AB de Villiers    414    253               667
8       RV Uthappa    481    182               663
9       KD Karthik    466    161               627
10        MS Dhoni    363    252               615


### 4.9 Boundary Percentage

In [98]:
boundary_pct = (
    merged_df.groupby(bat_col)
    .agg(total_runs=('batsman_runs', 'sum'),
         fours=('is_four', 'sum'),
         sixes=('is_six', 'sum'))
    .reset_index()
    .rename(columns={bat_col: 'batter'})
)
boundary_pct['boundary_runs'] = boundary_pct['fours'] * 4 + boundary_pct['sixes'] * 6
boundary_pct['boundary_percentage'] = (
    boundary_pct['boundary_runs'] / boundary_pct['total_runs'] * 100
).round(2)
boundary_pct = boundary_pct[boundary_pct['total_runs'] >= 500].sort_values('boundary_percentage', ascending=False).head(10)
print(boundary_pct[['batter', 'total_runs', 'boundary_runs', 'boundary_percentage']].to_string(index=False))

        batter  total_runs  boundary_runs  boundary_percentage
     SP Narine        1534           1238                80.70
    AD Russell        2488           1938                77.89
       PD Salt         653            500                76.57
      CH Gayle        4997           3786                75.77
 ST Jayasuriya         768            570                74.22
P Simran Singh         756            558                73.81
   YBK Jaiswal        1607           1180                73.43
  AC Gilchrist        2069           1508                72.89
      V Sehwag        2728           1972                72.29
   PC Valthaty         505            364                72.08


### 4.10 Dot Ball Analysis

In [99]:
dot_balls_total = merged_df['is_dot'].sum()
print(f'Total dot balls: {dot_balls_total}')

dot_balls_bowler = (
    merged_df.groupby(bowl_col)['is_dot']
    .sum()
    .reset_index()
    .rename(columns={bowl_col: 'bowler', 'is_dot': 'dot_balls'})
    .sort_values('dot_balls', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
dot_balls_bowler.index += 1
print('\nTop 10 bowlers by dot balls:')
print(dot_balls_bowler.to_string())

Total dot balls: 90438

Top 10 bowlers by dot balls:
             bowler  dot_balls
1           B Kumar       1632
2         SP Narine       1569
3          R Ashwin       1552
4         PP Chawla       1325
5   Harbhajan Singh       1263
6         JJ Bumrah       1228
7         RA Jadeja       1216
8         YS Chahal       1194
9          UT Yadav       1186
10         A Mishra       1185


### 4.11 Runs per Over Analysis

In [100]:
over_col = 'over' if 'over' in merged_df.columns else 'ball'

runs_per_over = (
    merged_df.groupby('over')['total_runs']
    .mean()
    .reset_index()
    .rename(columns={'total_runs': 'avg_runs_per_over'})
)
runs_per_over['avg_runs_per_over'] = runs_per_over['avg_runs_per_over'].round(2)
print(runs_per_over.to_string(index=False))

 over  avg_runs_per_over
    0               0.98
    1               1.17
    2               1.32
    3               1.36
    4               1.37
    5               1.37
    6               1.10
    7               1.19
    8               1.24
    9               1.22
   10               1.26
   11               1.29
   12               1.30
   13               1.34
   14               1.39
   15               1.43
   16               1.50
   17               1.59
   18               1.65
   19               1.78


### 4.12 Powerplay Performance (Overs 1–6)

In [101]:
powerplay_df = merged_df[merged_df['over'].between(1, 6)]

total_pp_runs = powerplay_df['total_runs'].sum()
print(f'Total Powerplay runs (all matches): {total_pp_runs}')

best_pp_teams = (
    powerplay_df.groupby(bat_team_col)['total_runs']
    .sum()
    .reset_index()
    .rename(columns={bat_team_col: 'team', 'total_runs': 'powerplay_runs'})
    .sort_values('powerplay_runs', ascending=False)
    .head(10)
)
print('\nBest teams in Powerplay:')
print(best_pp_teams.to_string(index=False))

Total Powerplay runs (all matches): 104405

Best teams in Powerplay:
                       team  powerplay_runs
             Mumbai Indians           12371
      Kolkata Knight Riders           11912
        Chennai Super Kings           11379
Royal Challengers Bangalore           10929
           Rajasthan Royals           10333
            Kings XI Punjab            9095
        Sunrisers Hyderabad            9047
           Delhi Daredevils            7506
             Delhi Capitals            4706
            Deccan Chargers            3407


### 4.13 Death Overs Performance (Overs 16–20)

In [102]:
death_df = merged_df[merged_df['over'].between(16, 20)]

death_team = (
    death_df.groupby(bat_team_col)['total_runs']
    .sum()
    .reset_index()
    .rename(columns={bat_team_col: 'team', 'total_runs': 'death_runs'})
    .sort_values('death_runs', ascending=False)
)

death_batter = (
    death_df.groupby(bat_col)['batsman_runs']
    .sum()
    .reset_index()
    .rename(columns={bat_col: 'batter', 'batsman_runs': 'death_runs'})
    .sort_values('death_runs', ascending=False)
    .head(10)
)

print('Best teams in Death Overs:')
print(death_team.head(10).to_string(index=False))
print('\nBest batters in Death Overs:')
print(death_batter.to_string(index=False))

Best teams in Death Overs:
                       team  death_runs
             Mumbai Indians        9598
        Chennai Super Kings        9061
Royal Challengers Bangalore        8417
      Kolkata Knight Riders        8053
           Rajasthan Royals        7281
        Sunrisers Hyderabad        6237
            Kings XI Punjab        6227
           Delhi Daredevils        5043
             Delhi Capitals        3141
            Deccan Chargers        2539

Best batters in Death Overs:
        batter  death_runs
      MS Dhoni        2786
    KA Pollard        1708
    KD Karthik        1565
AB de Villiers        1421
     RA Jadeja        1420
     RG Sharma        1176
     HH Pandya        1126
       V Kohli        1099
    AD Russell        1065
     DA Miller         988


In [103]:
# Save death_overs for export
death_overs = death_team.copy()

### 4.14 Run Distribution per Inning

In [104]:
inn_col = 'inning' if 'inning' in merged_df.columns else 'innings'

inning_dist = (
    merged_df.groupby([inn_col, 'over'])['total_runs']
    .mean()
    .reset_index()
    .rename(columns={'total_runs': 'avg_runs'})
)
inning_dist['avg_runs'] = inning_dist['avg_runs'].round(2)
print(inning_dist.pivot(index='over', columns=inn_col, values='avg_runs').to_string())

inning     1     2     3     4     5     6
over                                      
0       0.93  1.01  1.78  1.71  1.38  3.75
1       1.14  1.21   NaN   NaN   NaN   NaN
2       1.29  1.34   NaN   NaN   NaN   NaN
3       1.31  1.40   NaN   NaN   NaN   NaN
4       1.38  1.36   NaN   NaN   NaN   NaN
5       1.36  1.38   NaN   NaN   NaN   NaN
6       1.08  1.12   NaN   NaN   NaN   NaN
7       1.19  1.19   NaN   NaN   NaN   NaN
8       1.22  1.26   NaN   NaN   NaN   NaN
9       1.24  1.21   NaN   NaN   NaN   NaN
10      1.26  1.26   NaN   NaN   NaN   NaN
11      1.29  1.28   NaN   NaN   NaN   NaN
12      1.28  1.32   NaN   NaN   NaN   NaN
13      1.35  1.33   NaN   NaN   NaN   NaN
14      1.43  1.36   NaN   NaN   NaN   NaN
15      1.43  1.44   NaN   NaN   NaN   NaN
16      1.52  1.47   NaN   NaN   NaN   NaN
17      1.61  1.56   NaN   NaN   NaN   NaN
18      1.71  1.55   NaN   NaN   NaN   NaN
19      1.84  1.65   NaN   NaN   NaN   NaN


### 4.15 Toss Impact Analysis

In [105]:
if 'toss_winner' in matches_df.columns and 'winner' in matches_df.columns:
    toss_win = matches_df.copy()
    toss_win['toss_won_match'] = (toss_win['toss_winner'] == toss_win['winner']).astype(int)
    toss_advantage = toss_win['toss_won_match'].mean() * 100
    print(f'Toss winner also won the match: {toss_advantage:.1f}% of the time')

    toss_decision = toss_win.groupby('toss_decision')['toss_won_match'].mean() * 100
    print('\nWin % by toss decision:')
    print(toss_decision.round(1).to_string())
else:
    print('Toss columns not found in dataset.')

Toss winner also won the match: 50.6% of the time

Win % by toss decision:
toss_decision
bat      45.3
field    53.6


### 4.16 Player of Match Contribution

In [106]:
if 'player_of_match' in merged_df.columns:
    pom_runs = (
        merged_df.groupby(['match_id', bat_col])['batsman_runs']
        .sum()
        .reset_index()
    )
    pom_info = matches_df[['id', 'player_of_match']].rename(columns={'id': 'match_id'})
    pom_merged = pom_runs.merge(pom_info, on='match_id')
    pom_merged['is_pom'] = (pom_merged[bat_col] == pom_merged['player_of_match'])

    top_scorer_per_match = (
        pom_runs.sort_values('batsman_runs', ascending=False)
        .drop_duplicates(subset='match_id')
        .rename(columns={bat_col: 'top_scorer', 'batsman_runs': 'top_runs'})
    )
    pom_check = pom_info.merge(top_scorer_per_match, on='match_id')
    pom_check['pom_is_top_scorer'] = (pom_check['player_of_match'] == pom_check['top_scorer'])
    pct = pom_check['pom_is_top_scorer'].mean() * 100
    print(f'Player of Match was top run scorer in {pct:.1f}% of matches')
else:
    print('player_of_match column not found.')

Player of Match was top run scorer in 45.4% of matches


### 4.17 Venue-wise Analysis

In [107]:
venue_col = 'venue' if 'venue' in merged_df.columns else None

if venue_col:
    venue_matches = matches_df.groupby('venue')['id'].count().reset_index().rename(columns={'id': 'total_matches', 'venue': 'venue'})

    venue_runs = (
        merged_df.groupby(['match_id', venue_col])['total_runs']
        .sum()
        .reset_index()
        .groupby(venue_col)['total_runs']
        .mean()
        .round(1)
        .reset_index()
        .rename(columns={venue_col: 'venue', 'total_runs': 'avg_runs_per_match'})
    )
    venue_analysis = venue_matches.merge(venue_runs, on='venue').sort_values('total_matches', ascending=False)
    print(venue_analysis.head(10).to_string(index=False))
else:
    print('Venue column not available.')

                                     venue  total_matches  avg_runs_per_match
                              Eden Gardens             77               307.2
                          Wankhede Stadium             73               320.6
                     M Chinnaswamy Stadium             65               311.7
                          Feroz Shah Kotla             60               307.0
 Rajiv Gandhi International Stadium, Uppal             49               303.8
           MA Chidambaram Stadium, Chepauk             48               318.3
                    Sawai Mansingh Stadium             47               303.5
       Dubai International Cricket Stadium             46               314.1
                  Wankhede Stadium, Mumbai             45               346.4
Punjab Cricket Association Stadium, Mohali             35               313.9


### 4.18 City-wise Scoring Trends

In [108]:
if 'city' in merged_df.columns:
    city_runs = (
        merged_df.groupby(['match_id', 'city'])['total_runs']
        .sum()
        .reset_index()
        .groupby('city')['total_runs']
        .mean()
        .round(1)
        .reset_index()
        .rename(columns={'total_runs': 'avg_runs_per_match'})
        .sort_values('avg_runs_per_match', ascending=False)
    )
    print(city_runs.head(10).to_string(index=False))
else:
    print('City column not found.')

      city  avg_runs_per_match
 Bengaluru               360.3
  Guwahati               339.7
Dharamsala               339.4
    Mohali               335.0
    Rajkot               333.3
 Ahmedabad               331.0
    Mumbai               328.4
Chandigarh               325.6
   Cuttack               325.4
    Kanpur               324.5


### 4.19 Season-wise Run Trends

In [109]:
if 'season' in merged_df.columns:
    season_runs = (
        merged_df.groupby('season')['total_runs']
        .sum()
        .reset_index()
        .rename(columns={'total_runs': 'total_runs_in_season'})
        .sort_values('season')
    )
    print(season_runs.to_string(index=False))
else:
    print('Season column not found.')

 season  total_runs_in_season
2007/08                 17937
   2009                 16353
2009/10                 18883
   2011                 21154
   2012                 22453
   2013                 22602
   2014                 18931
   2015                 18353
   2016                 18862
   2017                 18786
   2018                 19901
   2019                 19434
2020/21                 19416
   2021                 18637
   2022                 24395
   2023                 25688
   2024                 25971


### 4.20 Winning Team Analysis

In [110]:
if 'winner' in matches_df.columns:
    win_count = (
        matches_df[matches_df['winner'] != 'Unknown']
        .groupby('winner')['id']
        .count()
        .reset_index()
        .rename(columns={'winner': 'team', 'id': 'wins'})
        .sort_values('wins', ascending=False)
    )
    print('Most match wins:')
    print(win_count.head(10).to_string(index=False))
else:
    print('Winner column not found.')

Most match wins:
                       team  wins
             Mumbai Indians   144
        Chennai Super Kings   138
      Kolkata Knight Riders   131
Royal Challengers Bangalore   116
           Rajasthan Royals   112
            Kings XI Punjab    88
        Sunrisers Hyderabad    88
           Delhi Daredevils    67
             Delhi Capitals    48
            Deccan Chargers    29


## Stage 5: Derived Insights

In [ ]:
print('KEY IPL INSIGHTS SUMMARY')

top_consistent = consistent_batters.iloc[0]
print(f"\n Most Consistent Batter  : {top_consistent['batter']}")
print(f"   Avg runs/match          : {top_consistent['avg_runs_per_match']}")
print(f"   Matches played          : {top_consistent['matches_played']}")

top_death_team = death_overs.iloc[0]
print(f"\n Best Death Overs Team   : {top_death_team['team']}")
print(f"   Total death-over runs   : {top_death_team['death_runs']}")

if venue_col and 'venue_analysis' in dir():
    top_venue = venue_analysis.sort_values('avg_runs_per_match', ascending=False).iloc[0]
    print(f"\n  High-Scoring Venue      : {top_venue['venue']}")
    print(f"   Avg runs/match          : {top_venue['avg_runs_per_match']}")

print(f"\n All-time Top Run Scorer  : {top_batters.iloc[0]['batter']}")
print(f"   Total IPL runs           : {top_batters.iloc[0]['total_runs']}")

print(f"\n Most Economic Bowler    : {economy.iloc[0]['bowler']}")
print(f"   Economy Rate             : {economy.iloc[0]['economy_rate']}")


KEY IPL INSIGHTS SUMMARY

 Most Consistent Batter  : DP Conway
   Avg runs/match          : 42.0
   Matches played          : 22

 Best Death Overs Team   : Mumbai Indians
   Total death-over runs   : 9598

  High-Scoring Venue      : Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam
   Avg runs/match          : 400.0

 All-time Top Run Scorer  : V Kohli
   Total IPL runs           : 8014

 Most Economic Bowler    : Sohail Tanvir
   Economy Rate             : 6.23


## Stage 6: Reporting — Clean DataFrames

In [112]:
print('\n--- Top 10 Batters ---')
print(top_batters.to_string())

print('\n--- Top 10 Bowlers by Economy ---')
print(economy[['bowler', 'runs_conceded', 'overs_bowled', 'economy_rate']].head(10).to_string(index=False))

print('\n--- Strike Rate (min 200 balls) ---')
print(strike_rate[['batter', 'total_runs', 'balls_faced', 'strike_rate']].head(10).to_string(index=False))


--- Top 10 Batters ---
            batter  total_runs
1          V Kohli        8014
2         S Dhawan        6769
3        RG Sharma        6630
4        DA Warner        6567
5         SK Raina        5536
6         MS Dhoni        5243
7   AB de Villiers        5181
8         CH Gayle        4997
9       RV Uthappa        4954
10      KD Karthik        4843

--- Top 10 Bowlers by Economy ---
         bowler  runs_conceded  overs_bowled  economy_rate
  Sohail Tanvir            275     44.166667          6.23
     A Chandila            245     39.000000          6.28
     FH Edwards            160     25.000000          6.40
SMSM Senanayake            211     32.500000          6.49
     SM Pollock            307     46.666667          6.58
       A Kumble           1089    163.833333          6.65
     GD McGrath            366     54.833333          6.67
 M Muralitharan           1765    263.500000          6.70
       IS Sodhi            204     30.333333          6.73
        J 

## Stage 7: Data Export

In [ ]:
runs_per_match.to_csv('output/runs_per_match.csv', index=False)
top_batters.to_csv('output/top_batters.csv', index=False)
strike_rate.to_csv('output/strike_rate.csv', index=False)
economy.to_csv('output/economy.csv', index=False)
team_scores.to_csv('output/team_scores.csv', index=False)
death_overs.to_csv('output/death_overs.csv', index=False)

print('CSV files saved to output')

CSV files saved to output


: 